# System 2: Single-Agent RAG (Showcase)

This notebook demonstrates **System 2 (Single-Agent RAG)**. 
Unlike System 1 (Monolith), which always follows a fixed `retrieve -> generate` pipeline, System 2 uses a **ReAct Agent** with access to specific tools.

The agent autonomously decides *when* to search, *how* to search, and can perform mathematical operations. Crucially, the **retrieval stack (chunking, vectorstore, hybrid index, reranker) is identical** to System 1 to ensure a fair architectural comparison.

## Available Tools:
1. `list_filings()`: Check which SEC 10-Ks are available in the knowledge base.
2. `search_section(ticker, fiscal_year, section)`: Target a specific section (e.g., MD&A) using ChromaDB metadata filtering.
3. `retrieve_chunks(query)`: Fallback to general hybrid semantic search across all docs.
4. `calculate(expression)`: Safe math evaluator (`simpleeval`) to prevent LLM math hallucinations.


In [4]:
import os
import sys
from pathlib import Path

# Ensure src/ is in the python path
if Path("src").exists():
    sys.path.append(os.path.abspath("."))
else:
    sys.path.append(os.path.abspath("../../"))

from dotenv import load_dotenv
load_dotenv()

from src.common.ingestion import ProcessedFiling
from src.systems.rag_agent.pipeline import AgentRAGPipeline

import warnings
warnings.filterwarnings('ignore')


## 1. Load Data & Build Pipeline
We load a small sample of parsed SEC 10-K filings and build the agent pipeline.
Behind the scenes, this:
1. Chunks the documents.
2. Builds the ChromaDB vectorstore (or loads if cached).
3. Instantiates the tools with injected dependencies.
4. Compiles the LangGraph ReAct agent.


In [5]:
# Load parsed JSON filings from disk
# Note: Ensure you have run the ingestion pipeline first!
from pathlib import Path
import os

# Dynamically resolve data directory regardless of whether the notebook 
# is running from the project root or the showcases folder
if Path("data/processed").exists():
    data_dir = Path("data/processed")
else:
    data_dir = Path("../../data/processed")

filings = []
if data_dir.exists():
    # Find .meta.json sidecars
    for meta_file in data_dir.rglob("*.meta.json"):
        md_file = meta_file.with_suffix("").with_suffix(".md")
        if md_file.exists():
            filings.append(ProcessedFiling.from_files(md_file, meta_file))
    
print(f"Loaded {len(filings)} filings from {data_dir.absolute()}")

# Initialize pipeline
print("Building Agent RAG Pipeline...")
pipeline = AgentRAGPipeline()
pipeline.build(filings)
print("Pipeline built successfully!")


Loaded 12 filings from c:\Users\Nutzer\Desktop\Uni\Master\Masterarbeit\agentic-financial-report-analysis\notebooks\showcases\..\..\data\processed
Building Agent RAG Pipeline...
Pipeline built successfully!


## 2. Single-Step Query (Targeted Retrieval)
Let's ask a question that requires pointing to a specific document section.
Notice how the agent uses `search_section` instead of generic retrieval.


In [6]:
query1 = "What are the key risk factors for Apple (AAPL) in FY2024?"
print(f"Query: {query1}\n")

res1 = pipeline.query(query1)

print("\n" + "="*50)
print(f"Answer:\n{res1.answer}")
print("="*50 + "\n")

print(f"Tokens Used: {res1.metrics.token_usage.total_tokens}")
print(f"Latency: {res1.metrics.latency_seconds:.2f}s")
print(f"Steps: {res1.metrics.num_steps}")

print("\nTool Call Log:")
for i, tc in enumerate(res1.tool_calls_log, 1):
    print(f"{i}. {tc['tool']} -> {tc['args']}")


Query: What are the key risk factors for Apple (AAPL) in FY2024?


Answer:
[{'type': 'text', 'text': "Based on Apple's (AAPL) 2024 10-K filing, the key risk factors include:\n\n*   **Innovation and R&D Investment:** The company's success depends on the timely introduction of innovative new products, services, and technologies. Significant R&D investments may not always yield expected returns, and there's no guarantee of successful development and marketing of new offerings (AAPL, 2024, Risk Factors).\n*   **Product Defects:** Apple's products and services can be affected by design and manufacturing defects, which could harm the company's business and reputation (AAPL, 2024, Risk Factors).\n*   **Antitrust Investigations and Litigation:** Apple is subject to antitrust investigations and litigation globally, including civil antitrust lawsuits in the U.S. regarding alleged monopolization in smartphone markets and investigations in Europe concerning App Store terms and conditions. These le

## 3. Multi-Step Query (Retrieval + Math)
Here we ask a question requiring the agent to find revenue numbers for multiple companies and then calculate the difference.


In [7]:
# Note: Edit the tickers below based on the filings you actually loaded
query2 = "What is the difference in total revenue between AAPL (FY2024) and MSFT (FY2024) in billions?"
print(f"Query: {query2}\n")

res2 = pipeline.query(query2)

print("\n" + "="*50)
print(f"Answer:\n{res2.answer}")
print("="*50 + "\n")

print(f"Tokens Used: {res2.metrics.token_usage.total_tokens}")
print(f"Latency: {res2.metrics.latency_seconds:.2f}s")
print(f"Steps: {res2.metrics.num_steps}")

print("\nTool Call Log:")
for i, tc in enumerate(res2.tool_calls_log, 1):
    print(f"{i}. {tc['tool']} -> {tc['args']}")


Query: What is the difference in total revenue between AAPL (FY2024) and MSFT (FY2024) in billions?



INFO:src.common.retrieval:FlashRank ranker loaded: ms-marco-MiniLM-L-12-v2
INFO:src.common.retrieval:FlashRank ranker loaded: ms-marco-MiniLM-L-12-v2
INFO:src.common.retrieval:Final retrieval: 4 documents
INFO:src.systems.rag_agent.tools.retrieve_chunks:retrieve_chunks: query='MSFT total revenue 2024' → 4 docs
INFO:src.common.retrieval:Final retrieval: 4 documents
INFO:src.systems.rag_agent.tools.retrieve_chunks:retrieve_chunks: query='AAPL total revenue 2024' → 4 docs
INFO:httpx:HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/master-thesis-489320/locations/global/publishers/google/models/gemini-embedding-001:predict "HTTP/1.1 200 OK"
INFO:src.common.retrieval:Hybrid retrieval: 23 documents
INFO:src.common.retrieval:Final retrieval: 4 documents
INFO:src.systems.rag_agent.tools.retrieve_chunks:retrieve_chunks: query='AAPL total net sales 2024' → 4 docs
INFO:src.systems.rag_agent.tools.calculate:calculate: '(391035 - 245122) / 1000' → 145.913
INFO:src.systems.rag_ag


Answer:
[{'type': 'text', 'text': "The difference in total revenue between Apple (AAPL) for fiscal year 2024 and Microsoft (MSFT) for fiscal year 2024 is $145.913 billion.\n\n*   AAPL's total net sales for FY2024 were $391,035 million (Source: AAPL FY2024 — Financial Statements).\n*   MSFT's total revenue for FY2024 was $245,122 million (Source: MSFT FY2024 — MD&A).", 'thought_signature': 'CucBAY89a18tiwk12N4aoLuRlQ6z1XcuwvcsHuJ4PeQx6Er75pBtAdv2IZ2X4PWLwjn8jesUkEbpK13nprFuuPfxgb8gitYEuNiWeI0Cl/d+i5ShgkQQdnpUjVk2rFHNE8EZFRzEWBrYtcvjvB7BXShJJE8jTbWhotwDM4izD9shWaRKJo62QIcDdG7cjqBQQWmOEgl2l00tK2M13LkK4uRIvbiy5xd1ZBeTPWzE74ecoK5D+QrUJd7bBflLY9/XwXT/O174fxbGaLZ1b4yrIgKIcYdiszhp3+Skcv5RwTAjBX/rvykdgoUv'}]

Tokens Used: 19184
Latency: 16.67s
Steps: 6

Tool Call Log:
1. search_section -> {'ticker': 'AAPL', 'fiscal_year': '2024', 'section': 'Financial Statements'}
2. search_section -> {'ticker': 'MSFT', 'fiscal_year': '2024', 'section': 'Financial Statements'}
3. retrieve_chunks -> {'query': '